In [12]:
# Faten TINZAGHTI  


# Extraction des séries GEO liées au récepteur androgène (AR) et aux expériences ChIP-seq

In [13]:
import requests               # Pour envoyer des requêtes HTTP vers les API NCBI
from bs4 import BeautifulSoup # Pour analyser les réponses XML d'ENTREZ
import pandas as pd           # Pour construire et sauvegarder le tableau final
import time                   # Pour respecter le délai entre les requêtes NCBI

# Paramètres : termes de recherche + email obligatoire pour ENTREZ

TERM = "ChIP-seq AND AR"  # Mot-clé de recherche sur GEO (CDH2 ChIP-seq)
EMAIL = "your_email@example.com"         # Peut rester comme ça, c'est accepté par NCBI

# URL de base pour toutes les API ENTREZ
API = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"


#  Fonction : recherche des séries GEO correspondant au terme donné

def entrez_search(term):
    """
    Recherche dans la base GEO (GDS) toutes les séries liées au terme demandé.
    Retourne une liste d'identifiants internes GDS.
    """

    url = API + "esearch.fcgi"  # API "esearch" pour interroger la base de données
    params = {
        "db": "gds",            # Base GEO DataSets
        "term": term,           # Terme de recherche
        "retmax": 5000,         # Nombre max d'entrées à renvoyer
        "retmode": "xml",       # Format de retour
        "email": EMAIL          # Email exigé par NCBI (mais pas vérifié)
    }

    r = requests.get(url, params=params)      # Envoi de la requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Analyse XML de la réponse

    # On extrait toutes les balises <Id> (identifiants internes GEO)
    ids = [id_tag.text for id_tag in soup.find_all("Id")]

    return ids  # Liste d'identifiants GDS bruts


# Fonction : récupération des informations détaillées d’un GSE

def fetch_gse_metadata(gds_id):
    """
    Récupère les métadonnées d'une série GEO via l'API esummary.
    Retourne un dictionnaire {GSE, Title, Platform, Samples}.
    Ignore les entrées incomplètes.
    """

    url = API + "esummary.fcgi"  # API "esummary" pour récupérer les infos détaillées
    params = {
        "db": "gds",             # Même base : GEO DataSets
        "id": gds_id,            # Identifiant GDS trouvé dans esearch
        "retmode": "xml",
        "email": EMAIL
    }

    r = requests.get(url, params=params)      # Requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Parsing XML

    item = soup.find("DocSum")                # Bloc principal des infos
    if item is None:
        return None  # Si vide → entrée invalidée

    # Petite fonction interne qui vérifie qu’un champ existe avant de l’extraire
    def safe_find(name):
        tag = item.find("Item", {"Name": name})
        return tag.text if tag else None

    # Extraction de l'identifiant GSE réel (ex: "GSE12345")
    acc = safe_find("Accession")
    if acc is None:
        return None  # Entrée corrompue → on ignore

    # Retour d’un dictionnaire structuré (sera ajouté au CSV)
    return {
        "GSE": acc,                     # Accession GSE
        "Title": safe_find("title"),    # Titre de l'étude
        "Platform": safe_find("platform"),  # Technologie (ex: Illumina)
        "Samples": safe_find("n_samples")   # Nombre d’échantillons
    }

# Programme principal : recherche + récupération + export CS

print("Recherche des séries via ENTREZ...")
ids = entrez_search(TERM)  # On lance la recherche de tous les GDS pertinents
print(f" {len(ids)} séries trouvées via ENTREZ.\n")

results = []  # Liste pour stocker les séries valides

# Boucle sur toutes les séries trouvées
for i, gds_id in enumerate(ids):
    print(f" Lecture série {i+1}/{len(ids)} : ID {gds_id} ...")

    data = fetch_gse_metadata(gds_id)  # Récupération des métadonnées
    if data is None:
        print("  Série ignorée (métadonnées incomplètes)")  # Problème → on saute
        continue

    results.append(data)  # Ajout à la liste des résultats valides

    time.sleep(0.34)  # Pause obligatoire pour respecter la limite NCBI (3 requêtes/sec)

# Export des résultat
print("\n----------------------------------")
print(f" Séries valides : {len(results)}")
print("----------------------------------")

df = pd.DataFrame(results)                      # Conversion en tableau pandas
df.to_csv("GEO_AR_ChIPseq_results.csv", index=False)  # Sauvegarde en CSV

print("Extraction terminée.")
print("Résultats enregistrés dans : GEO_AR_ChIPseq_results.csv")

Recherche des séries via ENTREZ...
 736 séries trouvées via ENTREZ.

 Lecture série 1/736 : ID 200315393 ...
 Lecture série 2/736 : ID 200329198 ...
 Lecture série 3/736 : ID 200305396 ...
 Lecture série 4/736 : ID 200305395 ...
 Lecture série 5/736 : ID 200305394 ...
 Lecture série 6/736 : ID 200305225 ...
 Lecture série 7/736 : ID 200296178 ...
 Lecture série 8/736 : ID 200289313 ...
 Lecture série 9/736 : ID 200315959 ...
 Lecture série 10/736 : ID 200241255 ...
 Lecture série 11/736 : ID 200301368 ...
 Lecture série 12/736 : ID 200301366 ...
 Lecture série 13/736 : ID 200261412 ...
 Lecture série 14/736 : ID 200318340 ...
 Lecture série 15/736 : ID 200227688 ...
 Lecture série 16/736 : ID 200306114 ...
 Lecture série 17/736 : ID 200292273 ...
 Lecture série 18/736 : ID 200313838 ...
 Lecture série 19/736 : ID 200295408 ...
 Lecture série 20/736 : ID 200308945 ...
 Lecture série 21/736 : ID 200296270 ...
 Lecture série 22/736 : ID 200293202 ...
 Lecture série 23/736 : ID 200293101 .

In [14]:
AR_chIPseq = pd.read_csv("GEO_AR_ChIPseq_results.csv")
AR_chIPseq = AR_chIPseq.drop("Platform", axis=1)

In [15]:
AR_chIPseq

,GSE,Title,Samples
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3
...,...,...,...
731,GSM1003798,Stanford_ChipSeq_MEL_ZNF-MIZD-CP1_(ab65767)_Ig...,0
732,GSM1003793,Stanford_ChipSeq_CH12_ZNF-MIZD-CP1_(ab65767)_I...,0
733,GSM1003611,Stanford_ChipSeq_K562_ZNF-MIZD-CP1_(ab65767)_I...,0
734,GSM712805,"AR ChIP-seq, 16h DHT treatment",0


In [16]:
AR_chIPseq.shape

(736, 3)

In [17]:
AR_chIPseq.isnull()

,GSE,Title,Samples
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
731,False,False,False
732,False,False,False
733,False,False,False
734,False,False,False


In [18]:
AR_chIPseq.isna()

,GSE,Title,Samples
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
731,False,False,False
732,False,False,False
733,False,False,False
734,False,False,False


# Extraction des séries GEO liées à CDH2 et aux expériences ChIP-seq

In [19]:
import requests               # Pour envoyer des requêtes HTTP vers les API NCBI
from bs4 import BeautifulSoup # Pour analyser les réponses XML d'ENTREZ
import pandas as pd           # Pour construire et sauvegarder le tableau final
import time                   # Pour respecter le délai entre les requêtes NCBI

# Paramètres : termes de recherche + email obligatoire pour ENTREZ

TERM = "ChIP-seq AND CDH2"  # Mot-clé de recherche sur GEO (CDH2 ChIP-seq)
EMAIL = "your_email@example.com"         # Peut rester comme ça, c'est accepté par NCBI

# URL de base pour toutes les API ENTREZ
API = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"


#  Fonction : recherche des séries GEO correspondant au terme donné

def entrez_search(term):
    """
    Recherche dans la base GEO (GDS) toutes les séries liées au terme demandé.
    Retourne une liste d'identifiants internes GDS.
    """

    url = API + "esearch.fcgi"  # API "esearch" pour interroger la base de données
    params = {
        "db": "gds",            # Base GEO DataSets
        "term": term,           # Terme de recherche
        "retmax": 5000,         # Nombre max d'entrées à renvoyer
        "retmode": "xml",       # Format de retour
        "email": EMAIL          # Email exigé par NCBI (mais pas vérifié)
    }

    r = requests.get(url, params=params)      # Envoi de la requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Analyse XML de la réponse

    # On extrait toutes les balises <Id> (identifiants internes GEO)
    ids = [id_tag.text for id_tag in soup.find_all("Id")]

    return ids  # Liste d'identifiants GDS bruts


# Fonction : récupération des informations détaillées d’un GSE

def fetch_gse_metadata(gds_id):
    """
    Récupère les métadonnées d'une série GEO via l'API esummary.
    Retourne un dictionnaire {GSE, Title, Platform, Samples}.
    Ignore les entrées incomplètes.
    """

    url = API + "esummary.fcgi"  # API "esummary" pour récupérer les infos détaillées
    params = {
        "db": "gds",             # Même base : GEO DataSets
        "id": gds_id,            # Identifiant GDS trouvé dans esearch
        "retmode": "xml",
        "email": EMAIL
    }

    r = requests.get(url, params=params)      # Requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Parsing XML

    item = soup.find("DocSum")                # Bloc principal des infos
    if item is None:
        return None  # Si vide → entrée invalidée

    # Petite fonction interne qui vérifie qu’un champ existe avant de l’extraire
    def safe_find(name):
        tag = item.find("Item", {"Name": name})
        return tag.text if tag else None

    # Extraction de l'identifiant GSE réel (ex: "GSE12345")
    acc = safe_find("Accession")
    if acc is None:
        return None  # Entrée corrompue → on ignore

    # Retour d’un dictionnaire structuré (sera ajouté au CSV)
    return {
        "GSE": acc,                     # Accession GSE
        "Title": safe_find("title"),    # Titre de l'étude
        "Platform": safe_find("platform"),  # Technologie (ex: Illumina)
        "Samples": safe_find("n_samples")   # Nombre d’échantillons
    }

# Programme principal : recherche + récupération + export CS

print("Recherche des séries via ENTREZ...")
ids = entrez_search(TERM)  # On lance la recherche de tous les GDS pertinents
print(f" {len(ids)} séries trouvées via ENTREZ.\n")

results = []  # Liste pour stocker les séries valides

# Boucle sur toutes les séries trouvées
for i, gds_id in enumerate(ids):
    print(f" Lecture série {i+1}/{len(ids)} : ID {gds_id} ...")

    data = fetch_gse_metadata(gds_id)  # Récupération des métadonnées
    if data is None:
        print("  Série ignorée (métadonnées incomplètes)")  # Problème → on saute
        continue

    results.append(data)  # Ajout à la liste des résultats valides

    time.sleep(0.34)  # Pause obligatoire pour respecter la limite NCBI (3 requêtes/sec)

# Export des résultat
print("\n----------------------------------")
print(f" Séries valides : {len(results)}")
print("----------------------------------")

df = pd.DataFrame(results)                      # Conversion en tableau pandas
df.to_csv("GEO_CDH2_ChIPseq_results.csv", index=False)  # Sauvegarde en CSV

print("Extraction terminée.")
print("Résultats enregistrés dans : GEO_CDH2_ChIPseq_results.csv")

Recherche des séries via ENTREZ...
 5 séries trouvées via ENTREZ.

 Lecture série 1/5 : ID 200267053 ...
 Lecture série 2/5 : ID 200266928 ...
 Lecture série 3/5 : ID 200239415 ...
 Lecture série 4/5 : ID 200239414 ...
 Lecture série 5/5 : ID 200113279 ...

----------------------------------
📊 Séries valides : 5
----------------------------------
Extraction terminée.
Résultats enregistrés dans : GEO_CDH2_ChIPseq_results.csv


In [20]:
CDH2_chIPseq = pd.read_csv("GEO_CDH2_ChIPseq_results.csv")
CDH2_chIPseq = CDH2_chIPseq.drop("Platform", axis=1)

In [21]:
CDH2_chIPseq

,GSE,Title,Samples
0,GSE267053,Rescuing DNMT1 Fails to Fully Reverse the Mole...,96
1,GSE266928,Rescuing DNMT1 Fails to Fully Reverse the Mole...,48
2,GSE239415,SAMD1 suppresses epithelial-mesenchymal transi...,6
3,GSE239414,SAMD1 suppresses epithelial-mesenchymal transi...,6
4,GSE113279,WDR5 regulates EMT and metastasis in breast ca...,5


In [22]:
CDH2_chIPseq.shape

(5, 3)

In [23]:
CDH2_chIPseq.isna()

,GSE,Title,Samples
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False


In [24]:
CDH2_chIPseq.isna()

,GSE,Title,Samples
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False


# Extraction des séries GEO liées à N_cadhérine et aux expériences ChIP-seq

In [25]:
import requests               # Pour envoyer des requêtes HTTP vers les API NCBI
from bs4 import BeautifulSoup # Pour analyser les réponses XML d'ENTREZ
import pandas as pd           # Pour construire et sauvegarder le tableau final
import time                   # Pour respecter le délai entre les requêtes NCBI

# Paramètres : termes de recherche + email obligatoire pour ENTREZ

TERM = "ChIP-seq AND N_cadherin"  # Mot-clé de recherche sur GEO (AR ChIP-seq)
EMAIL = "your_email@example.com"         # Peut rester comme ça, c'est accepté par NCBI

# URL de base pour toutes les API ENTREZ
API = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"


#  Fonction : recherche des séries GEO correspondant au terme donné

def entrez_search(term):
    """
    Recherche dans la base GEO (GDS) toutes les séries liées au terme demandé.
    Retourne une liste d'identifiants internes GDS.
    """

    url = API + "esearch.fcgi"  # API "esearch" pour interroger la base de données
    params = {
        "db": "gds",            # Base GEO DataSets
        "term": term,           # Terme de recherche
        "retmax": 5000,         # Nombre max d'entrées à renvoyer
        "retmode": "xml",       # Format de retour
        "email": EMAIL          # Email exigé par NCBI (mais pas vérifié)
    }

    r = requests.get(url, params=params)      # Envoi de la requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Analyse XML de la réponse

    # On extrait toutes les balises <Id> (identifiants internes GEO)
    ids = [id_tag.text for id_tag in soup.find_all("Id")]

    return ids  # Liste d'identifiants GDS bruts


# Fonction : récupération des informations détaillées d’un GSE

def fetch_gse_metadata(gds_id):
    """
    Récupère les métadonnées d'une série GEO via l'API esummary.
    Retourne un dictionnaire {GSE, Title, Platform, Samples}.
    Ignore les entrées incomplètes.
    """

    url = API + "esummary.fcgi"  # API "esummary" pour récupérer les infos détaillées
    params = {
        "db": "gds",             # Même base : GEO DataSets
        "id": gds_id,            # Identifiant GDS trouvé dans esearch
        "retmode": "xml",
        "email": EMAIL
    }

    r = requests.get(url, params=params)      # Requête HTTP
    soup = BeautifulSoup(r.text, "xml")       # Parsing XML

    item = soup.find("DocSum")                # Bloc principal des infos
    if item is None:
        return None  # Si vide → entrée invalidée

    # Petite fonction interne qui vérifie qu’un champ existe avant de l’extraire
    def safe_find(name):
        tag = item.find("Item", {"Name": name})
        return tag.text if tag else None

    # Extraction de l'identifiant GSE réel (ex: "GSE12345")
    acc = safe_find("Accession")
    if acc is None:
        return None  # Entrée corrompue → on ignore

    # Retour d’un dictionnaire structuré (sera ajouté au CSV)
    return {
        "GSE": acc,                     # Accession GSE
        "Title": safe_find("title"),    # Titre de l'étude
        "Platform": safe_find("platform"),  # Technologie (ex: Illumina)
        "Samples": safe_find("n_samples")   # Nombre d’échantillons
    }

# Programme principal : recherche + récupération + export CS

print("Recherche des séries via ENTREZ...")
ids = entrez_search(TERM)  # On lance la recherche de tous les GDS pertinents
print(f" {len(ids)} séries trouvées via ENTREZ.\n")

results = []  # Liste pour stocker les séries valides

# Boucle sur toutes les séries trouvées
for i, gds_id in enumerate(ids):
    print(f" Lecture série {i+1}/{len(ids)} : ID {gds_id} ...")

    data = fetch_gse_metadata(gds_id)  # Récupération des métadonnées
    if data is None:
        print("  Série ignorée (métadonnées incomplètes)")  # Problème → on saute
        continue

    results.append(data)  # Ajout à la liste des résultats valides

    time.sleep(0.34)  # Pause obligatoire pour respecter la limite NCBI (3 requêtes/sec)

# Export des résultat
print("\n----------------------------------")
print(f" Séries valides : {len(results)}")
print("----------------------------------")

df = pd.DataFrame(results)                      # Conversion en tableau pandas
df.to_csv("GEO_N_cadherin_ChIPseq_results.csv", index=False)  # Sauvegarde en CSV
print("Extraction terminée.")
print("Résultats enregistrés dans : GEO_N_cadherin_ChIPseq_results.csv")

Recherche des séries via ENTREZ...
 68 séries trouvées via ENTREZ.

 Lecture série 1/68 : ID 200308417 ...
 Lecture série 2/68 : ID 200308183 ...
 Lecture série 3/68 : ID 200286440 ...
 Lecture série 4/68 : ID 200237500 ...
 Lecture série 5/68 : ID 200248092 ...
 Lecture série 6/68 : ID 200231753 ...
 Lecture série 7/68 : ID 200239415 ...
 Lecture série 8/68 : ID 200239414 ...
 Lecture série 9/68 : ID 200242918 ...
 Lecture série 10/68 : ID 200227823 ...
 Lecture série 11/68 : ID 200223531 ...
 Lecture série 12/68 : ID 200210523 ...
 Lecture série 13/68 : ID 200215991 ...
 Lecture série 14/68 : ID 200183407 ...
 Lecture série 15/68 : ID 200152365 ...
 Lecture série 16/68 : ID 200152364 ...
 Lecture série 17/68 : ID 200152363 ...
 Lecture série 18/68 : ID 200152362 ...
 Lecture série 19/68 : ID 200193165 ...
 Lecture série 20/68 : ID 200184856 ...
 Lecture série 21/68 : ID 200154277 ...
 Lecture série 22/68 : ID 200185682 ...
 Lecture série 23/68 : ID 200126897 ...
 Lecture série 24/68 

In [26]:
N_cadherin_chIPseq = pd.read_csv("GEO_N_cadherin_ChIPseq_results.csv")
N_cadherin_chIPseq = N_cadherin_chIPseq.drop("Platform", axis=1)

In [27]:
N_cadherin_chIPseq

,GSE,Title,Samples
0,GSE308417,Fat cadherin cleavage releases a transcription...,34
1,GSE308183,Fat cadherin cleavage releases a transcription...,6
2,GSE286440,Endothelial KLF15/VASN axis inhibits angiogene...,6
3,GSE237500,E-cadherin regulates female tumor aggressivene...,13
4,GSE248092,Tubular Foxp2 promotes kidney fibrosis by regu...,2
...,...,...,...
63,GSM2698833,HIF-1a ChIP-seq,0
64,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0
65,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0
66,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0


# Identification et extraction des données en commun entre AR_chIPseq et N_cadherin_chIPseq

In [28]:
matches = N_cadherin_chIPseq['GSE'].isin(AR_chIPseq['GSE'])
matches

0     False
1     False
2     False
3     False
4     False
      ...  
63    False
64    False
65    False
66    False
67    False
Name: GSE, Length: 68, dtype: bool

In [30]:
AR_N_cadherin_chIPseq = AR_chIPseq[AR_chIPseq["GSE"].isin(N_cadherin_chIPseq["GSE"])]
#AR_N_cadherin_chIPseq = AR_N_cadherin_chIPseq.drop("Platform", axis=1)
AR_N_cadherin_chIPseq

,GSE,Title,Samples
297,GSE92574,Mapping of DHT-responsive or -independent AR-b...,16
298,GSE92347,Mapping of DHT-responsive or -independent AR-b...,9
342,GSE51334,DNA replication-timing boundaries separate sta...,993
389,GSE32465,Transcription Factor Binding Sites by ChIP-seq...,399
393,GSE31477,ENCODE Transcription Factor Binding Sites by C...,426


# Identification et extraction des données en commun entre CDH2_chIPseq et N_cadherin_chIPseq

In [31]:
matches = CDH2_chIPseq['GSE'].isin(N_cadherin_chIPseq['GSE'])
matches

0    False
1    False
2     True
3     True
4    False
Name: GSE, dtype: bool

In [32]:
CDH2_N_cadherin_chIPseq = CDH2_chIPseq[CDH2_chIPseq["GSE"].isin(N_cadherin_chIPseq["GSE"])]
CDH2_N_cadherin_chIPseq

,GSE,Title,Samples
2,GSE239415,SAMD1 suppresses epithelial-mesenchymal transi...,6
3,GSE239414,SAMD1 suppresses epithelial-mesenchymal transi...,6


# Identification et extraction des données en commun entre CDH2_chIPseq et AR_chIPseq

In [33]:
matches= CDH2_chIPseq['GSE'].isin(AR_chIPseq['GSE'])
matches

0    False
1    False
2    False
3    False
4    False
Name: GSE, dtype: bool

# Fusion des jeux de données ChIP-seq et suppression des doublons

In [35]:
DF_commune = pd.concat([AR_chIPseq, CDH2_chIPseq , N_cadherin_chIPseq ], axis=0)
#DF_commune = DF_commune.drop(columns="Platform") 
DF_commune = DF_commune.drop_duplicates() 

In [36]:
DF_commune

,GSE,Title,Samples
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3
...,...,...,...
63,GSM2698833,HIF-1a ChIP-seq,0
64,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0
65,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0
66,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0


# Ajout des liens directs vers GEO

In [37]:
DF_commune['link'] = DF_commune['GSE'].apply(
    lambda x: f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={x}"
)
DF_commune

,GSE,Title,Samples,link
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
...,...,...,...,...
63,GSM2698833,HIF-1a ChIP-seq,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
64,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
65,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
66,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...


In [38]:
DF_commune.to_csv('Ready_for_now.csv', index=False) 

In [39]:
DF_commune = pd.read_csv("Ready_for_now.csv")
DF_commune

,GSE,Title,Samples,link
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
...,...,...,...,...
797,GSM2698833,HIF-1a ChIP-seq,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
798,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
799,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
800,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...


In [40]:
from Bio import Entrez
import time

# Always provide your email to NCBI
Entrez.email = "d.ivanova@example.com"  # replace with your real email

# Load your dataframe
df = DF_commune

abstracts = []
dates = []

for i, gse_id in enumerate(DF_commune["GSE"]):
    print(f"Processing {i+1}/{len(DF_commune)}: {gse_id}")
    try:
        # Step 1: Search PubMed for the GSE ID
        search_handle = Entrez.esearch(db="pubmed", term=f"{gse_id}[All Fields]")
        search_results = Entrez.read(search_handle)
        search_handle.close()

        pmids = search_results["IdList"]

        if pmids:
            # Step 2: Fetch the first result
            fetch_handle = Entrez.efetch(db="pubmed", id=pmids[0], rettype="xml", retmode="xml")
            record = Entrez.read(fetch_handle)
            fetch_handle.close()

            article = record["PubmedArticle"][0]["MedlineCitation"]["Article"]

            # Get abstract
            abstract = article.get("Abstract", {}).get("AbstractText", ["N/A"])
            abstract_text = " ".join([str(a) for a in abstract])

            # Get date
            pub_date = article.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
            date_str = f"{pub_date.get('Year', '')} {pub_date.get('Month', '')} {pub_date.get('Day', '')}".strip()

        else:
            abstract_text = "Not found"
            date_str = "Not found"

    except Exception as e:
        abstract_text = f"Error: {e}"
        date_str = "Error"

    abstracts.append(abstract_text)
    dates.append(date_str)

    time.sleep(0.4)  # Be polite to NCBI servers

# Add results to dataframe
df["Abstract"] = abstracts
df["Publication_Date"] = dates

# Save
df.to_excel("PUB_MED_abstracts.xlsx", index=False)
print("Done! Saved to Ready_for_now_with_abstracts.xlsx")

Processing 1/802: GSE315393
Processing 2/802: GSE329198
Processing 3/802: GSE305396
Processing 4/802: GSE305395
Processing 5/802: GSE305394
Processing 6/802: GSE305225
Processing 7/802: GSE296178
Processing 8/802: GSE289313
Processing 9/802: GSE315959
Processing 10/802: GSE241255
Processing 11/802: GSE301368
Processing 12/802: GSE301366
Processing 13/802: GSE261412
Processing 14/802: GSE318340
Processing 15/802: GSE227688
Processing 16/802: GSE306114
Processing 17/802: GSE292273


KeyboardInterrupt: 

In [41]:
import pandas as pd
from Bio import Entrez
import time

Entrez.email = "d.ivanova@example.com"  # replace with your real email

abstracts = []
dates = []

def fetch_pubmed(term):
    """Search PubMed with a given term, return abstract and date or None"""
    search_handle = Entrez.esearch(db="pubmed", term=term)
    search_results = Entrez.read(search_handle)
    search_handle.close()
    
    pmids = search_results["IdList"]
    
    if pmids:
        fetch_handle = Entrez.efetch(db="pubmed", id=pmids[0], rettype="xml", retmode="xml")
        record = Entrez.read(fetch_handle)
        fetch_handle.close()
        
        article = record["PubmedArticle"][0]["MedlineCitation"]["Article"]
        
        abstract = article.get("Abstract", {}).get("AbstractText", ["N/A"])
        abstract_text = " ".join([str(a) for a in abstract])
        
        pub_date = article.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
        date_str = f"{pub_date.get('Year', '')} {pub_date.get('Month', '')} {pub_date.get('Day', '')}".strip()
        
        return abstract_text, date_str
    
    return None, None

for i, row in enumerate(DF_commune.itertuples()):
    print(f"Processing {i+1}/{len(DF_commune)}: {row.GSE}")
    
    try:
        # Try 1: search by GSE ID
        abstract_text, date_str = fetch_pubmed(f"{row.GSE}[All Fields]")
        
        # Try 2: fallback to full title
        if abstract_text is None:
            print(f"  → Not found by GSE ID, trying full title...")
            abstract_text, date_str = fetch_pubmed(row.Title)
        
        # Try 3: fallback to shortened title
        if abstract_text is None:
            print(f"  → Trying shortened title...")
            short_title = " ".join(row.Title.split()[:8])
            abstract_text, date_str = fetch_pubmed(short_title)
        
        # If still not found after all 3 tries
        if abstract_text is None:
            abstract_text = "Not found"
            date_str = "Not found"
            print(f"  → Not found by any method")
        else:
            print(f"  → Found!")

    except Exception as e:
        abstract_text = f"Error: {e}"
        date_str = "Error"
        print(f"  → Error: {e}")

    abstracts.append(abstract_text)
    dates.append(date_str)
    time.sleep(0.4)

DF_commune["Abstract"] = abstracts
DF_commune["Publication_Date"] = dates

DF_commune.to_excel("Ready_for_now_with_abstracts.xlsx", index=False)
print("Done!")

Processing 1/802: GSE315393
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Not found by any method
Processing 2/802: GSE329198
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Not found by any method
Processing 3/802: GSE305396
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Not found by any method
Processing 4/802: GSE305395
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Not found by any method
Processing 5/802: GSE305394
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Not found by any method
Processing 6/802: GSE305225
  → Not found by GSE ID, trying full title...
  → Found!
Processing 7/802: GSE296178
  → Not found by GSE ID, trying full title...
  → Found!
Processing 8/802: GSE289313
  → Not found by GSE ID, trying full title...
  → Trying shortened title...
  → Found!
Processing 9/802: GSE315959
  → Not found by GSE ID, tr

In [46]:
not_found = (DF_commune["Abstract"]=="Not found").sum()
total = len(DF_commune)
found = total - not_found
print(f"Found:     {found}/{total}")
print(f"Not found: {not_found}/{total}")

Found:     403/802
Not found: 399/802


In [1]:
!pip install keybert

   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   - -------------------------------------- 3.1/123.0 MB 15.1 MB/s eta 0:00:08
   - -------------------------------------- 6.0/123.0 MB 14.3 MB/s eta 0:00:09
   --- ------------------------------------ 10.5/123.0 MB 16.4 MB/s eta 0:00:07
   ---- ----------------------------------- 15.2/123.0 MB 18.0 MB/s eta 0:00:06
   ------ --------------------------------- 20.4/123.0 MB 19.4 MB/s eta 0:00:06
   -------- ------------------------------- 25.4/123.0 MB 20.2 MB/s eta 0:00:05
   ---------- ----------------------------- 31.7/123.0 MB 21.3 MB/s eta 0:00:05
   ------------ --------------------------- 38.0/123.0 MB 22.4 MB/s eta 0:00:04
   -------------- ------------------------- 43.5/123.0 MB 23.0 MB/s eta 0:00:04
   ---------------- ----------------------- 50.1/123.0 MB 23.9 

In [10]:
import pandas as pd
DF_commune = pd.read_csv("Ready_for_now_with_abstracts.csv", encoding="latin-1")
DF_commune

,GSE,Title,Samples,link,Abstract PUB MED,Publication_Date
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
...,...,...,...,...,...,...
797,GSM2698833,HIF-1a ChIP-seq,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Clear cell renal cell carcinoma (ccRCC) is pri...,2026 Apr 17
798,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
799,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found
800,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found


## Ici le code permet d'ajouter les abstraits depuis PUB MED

In [13]:
from keybert import KeyBERT

# Load the model
kw_model = KeyBERT()

def extract_keywords(abstract, n=5):
    # Skip empty/NaN/not found values
    if pd.isna(abstract):
        return []
    # Skip rows with no abstract
    if abstract in ["Not found", "N/A"] or "Error" in str(abstract):
        return []
    
    keywords = kw_model.extract_keywords(
        abstract,
        keyphrase_ngram_range=(1, 2),  # single words and 2-word phrases
        stop_words='english',           # ignore common words like "the", "and"
        top_n=n                         # number of keywords to extract
    )
    return [kw[0] for kw in keywords]  # return just the words, not the scores

# Apply to all abstracts
print("Extracting keywords...")
DF_commune["Keywords"] = DF_commune["Abstract PUB MED"].apply(extract_keywords)
print("Done!")

# Preview results
print(DF_commune[["GSE", "Keywords"]].head(10))

Extracting keywords...
Done!
         GSE                                           Keywords
0  GSE315393                                                 []
1  GSE329198                                                 []
2  GSE305396                                                 []
3  GSE305395                                                 []
4  GSE305394                                                 []
5  GSE305225  [h3k9me3 seeds, seed dormancy, seeds arabidops...
6  GSE296178  [androgen receptor, prostate lineage, hormonal...
7  GSE289313  [pathogenesis psoriasis, kynureninase psoriasi...
8  GSE315959  [foxa1 binding, foxa1 pioneering, androgen rec...
9  GSE241255  a chromatin, lncrna chromatin, methyltransf...


In [14]:
DF_commune

,GSE,Title,Samples,link,Abstract PUB MED,Publication_Date,Keywords
0,GSE315393,Recurrent FOXA1 mutation redirects NIPBL chrom...,12,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
1,GSE329198,PGC-1alpha pathway dysregulation disrupts myof...,14,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
2,GSE305396,PGC-1alpha pathway dysregulation disrupts myof...,6,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
3,GSE305395,PGC-1alpha pathway dysregulation disrupts myof...,27,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
4,GSE305394,PGC-1alpha pathway dysregulation disrupts myof...,3,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
...,...,...,...,...,...,...,...
797,GSM2698833,HIF-1a ChIP-seq,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Clear cell renal cell carcinoma (ccRCC) is pri...,2026 Apr 17,"[micrornas implicated, micrornas, expression m..."
798,GSM1010729,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
799,GSM730632,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]
800,GSM730631,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",0,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Not found,Not found,[]


In [16]:
DF_commune2 = pd.read_csv("GEO_GSE_summary_corrected.csv", encoding="latin-1")
DF_commune2

,GSE,Samples,Title,Design_GSE,Summary_GSE,Title_GSE
0,GSE315959,51,Dissecting FOXA1 pioneering function by acute ...,NaN,Pioneer factors control transcription by openi...,Dissecting FOXA1 pioneering function by acute ...
1,GSE241255,4,Chromatin immunoprecipitation DNA sequencing (...,NaN,Purpose: Zinc Finger MIZ-Type Containing 1 (Zm...,Chromatin immunoprecipitation DNA sequencing (...
2,GSE301368,10,Acute hormonal signaling-induced 3D chromatin ...,NaN,Recent molecular advancements have revolutioni...,Acute hormonal signaling-induced 3D chromatin ...
3,GSE301366,24,Acute hormonal signaling-induced 3D chromatin ...,NaN,Recent molecular advancements have revolutioni...,Acute hormonal signaling-induced 3D chromatin ...
4,GSE261412,24,Acute hormonal signaling-induced 3D chromatin ...,NaN,Î²3-adrenergic receptor (Î²3-AR) hormonal sign...,Acute hormonal signaling-induced 3D chromatin ...
...,...,...,...,...,...,...
786,GSM2698833,0,HIF-1a ChIP-seq,NaN,This SuperSeries is composed of the SubSeries ...,Open Chromatin and HIF-binding in renal tubula...
787,GSM1010729,0,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,NaN,Eukaryotic chromosomes replicate in a temporal...,DNA replication-timing boundaries separate sta...
788,GSM730632,0,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",NaN,This SuperSeries is composed of the SubSeries ...,The transcriptional program controlled by Runx...
789,GSM730631,0,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",NaN,This SuperSeries is composed of the SubSeries ...,The transcriptional program controlled by Runx...


In [25]:
# Check column names exactly
print(DF_commune2.columns.tolist())

# Check how many are GSE vs GSM
print("GSE entries:", DF_commune2["GSE"].str.startswith("GSE").sum())
print("GSM entries:", DF_commune2["GSE"].str.startswith("GSM").sum())

# Check how many Summary_GSE are filled
print("Summaries filled:", DF_commune2["Summary_GSE"].notna().sum())

['GSE', 'Samples', 'Title', 'Design_GSE', 'Summary_GSE', 'Title_GSE']
GSE entries: 449
GSM entries: 342
Summaries filled: 791


In [26]:
from keybert import KeyBERT
import pandas as pd

kw_model = KeyBERT()

def extract_keywords(text, n=5):
    if pd.isna(text):
        return []
    if text in ["Not found", "N/A"] or "Error" in str(text):
        return []
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=n
    )
    return [kw[0] for kw in keywords]

print("Extracting keywords...")
DF_commune2["Keywords"] = DF_commune2["Summary_GSE"].apply(extract_keywords)
print("Done!")
print(DF_commune2[["GSE", "Keywords"]].head())

Extracting keywords...
Done!
         GSE                                           Keywords
0  GSE315959  [genes foxa1, foxa1 binding, foxa1 pioneering,...
1  GSE241255  [sequencing zmiz1, zmiz1 associated, containin...
2  GSE301368  [adipocytes, brown adipocytes, receptor î²3, a...
3  GSE301366  [adipocytes, brown adipocytes, receptor î²3, a...
4  GSE261412  [adipocytes implications, 3d genome, receptor ...


In [27]:
DF_commune2

,GSE,Samples,Title,Design_GSE,Summary_GSE,Title_GSE,Keywords
0,GSE315959,51,Dissecting FOXA1 pioneering function by acute ...,NaN,Pioneer factors control transcription by openi...,Dissecting FOXA1 pioneering function by acute ...,"[genes foxa1, foxa1 binding, foxa1 pioneering,..."
1,GSE241255,4,Chromatin immunoprecipitation DNA sequencing (...,NaN,Purpose: Zinc Finger MIZ-Type Containing 1 (Zm...,Chromatin immunoprecipitation DNA sequencing (...,"[sequencing zmiz1, zmiz1 associated, containin..."
2,GSE301368,10,Acute hormonal signaling-induced 3D chromatin ...,NaN,Recent molecular advancements have revolutioni...,Acute hormonal signaling-induced 3D chromatin ...,"[adipocytes, brown adipocytes, receptor î²3, a..."
3,GSE301366,24,Acute hormonal signaling-induced 3D chromatin ...,NaN,Recent molecular advancements have revolutioni...,Acute hormonal signaling-induced 3D chromatin ...,"[adipocytes, brown adipocytes, receptor î²3, a..."
4,GSE261412,24,Acute hormonal signaling-induced 3D chromatin ...,NaN,Î²3-adrenergic receptor (Î²3-AR) hormonal sign...,Acute hormonal signaling-induced 3D chromatin ...,"[adipocytes implications, 3d genome, receptor ..."
...,...,...,...,...,...,...,...
786,GSM2698833,0,HIF-1a ChIP-seq,NaN,This SuperSeries is composed of the SubSeries ...,Open Chromatin and HIF-binding in renal tubula...,"[superseries composed, subseries listed, super..."
787,GSM1010729,0,HudsonAlpha_ChipSeq_GM12878_MTA3_(SC-81325)_v0...,NaN,Eukaryotic chromosomes replicate in a temporal...,DNA replication-timing boundaries separate sta...,"[chromosomes replicate, regions chromatin, chr..."
788,GSM730632,0,"Runx1+/VE-cadherin-/CD41+, ChIP-seq",NaN,This SuperSeries is composed of the SubSeries ...,The transcriptional program controlled by Runx...,"[superseries composed, subseries listed, super..."
789,GSM730631,0,"Runx1+/VE-cadherin+/CD41-, ChIP-seq",NaN,This SuperSeries is composed of the SubSeries ...,The transcriptional program controlled by Runx...,"[superseries composed, subseries listed, super..."
